In [ ]:
# Data extraction from MPC is done with the MPC package in astroquery
from astroquery.mpc import MPC

In [ ]:
# To get orbitting data from asteroids use query_object(s) (plural or singular).

# You can search one asteroid by name:
by_name = MPC.query_object('asteroid', 'ceres') # Search Ceres

# You can search multiple asteroids by limiting the parameter
by_max = MPC.query_objects('asteroid', inclination_min=170) # All asteroids with 170 of inclination or more
by_min = MPC.query_objects('asteroid', absolute_magnitude_max=0) # All asteroids with with absolute magnitude of 0 or more 

# You can search for asteroids that have data in a parameter, also, you can limit the amount of asteroids returned
limited = MPC.query_objects('asteroid', name='is_not_null', limit=10) # Any asteroid with a name, returns exactly 10

# You can search for asteroids in a sorted manner
by_sort_desc = MPC.query_objects('asteroid', order_by_desc='semimajor_axes', limit=10) # 10 asteroids with highest semimajor axes (10 furthest asteroids)

# You can limit the returning parameters to only a few
limited_params = MPC.query_object('asteroid', name='ceres', return_fields='name, number') # Returns only the name and number of Ceres

16384 asteroids extracted.


In [ ]:
# To get ephemerides from asteroids use get_ephemerides

# You can search by name, designation or number
eph_by_name = MPC.get_ephemeris('ceres') # Ceres ephemerides
eph_by_desi = MPC.get_ephemeris('1927 LA') # Asteroid without a name
eph_by_code = MPC.get_ephemeris('24') # Themis ephemerides

# You can specify the steps of the ephemerides using the astropy's quantity
eph_with_step = MPC.get_ephemeris('ceres', step='1h') # From today, return ephemerides every hour till the default ending (not clear)

# You can specify the amount of ephemerides you want to receive as well as the starting date (the latter using astropy's time format)
eph_with_amount = MPC.get_ephemeris('ceres', step='1d', number=10, start='2020-01-01') # Returns 10 ephemerides of ceres with 1d between each starting in 01/01/2020

# You can specify the observer's location using an IAU observatory code; an array of long, lat, and alt; or an EarthLocation
eph_obs_code = MPC.get_ephemeris('ceres', location='G37') # Discovery channel telescope (code=G37)
eph_obs_coor = MPC.get_ephemeris('ceres', location=('24d', '-22d', '1000m')) # Botswana telescope (long=24d, lat=-22d, alt=1000m)

# You can get the earth locations of all observatories using get_observatory_codes
obs = MPC.get_observatory_codes() # If you don't want them to be cached, you can use 'cache=False'

# Values in the eph tables have units, you can convert them to other units using astropy quantities
eph_with_amount['Proper motion'].quantity.to('deg/h').max() # Converts the maximum proper motion in the ephemerides to deg/h

In [28]:
a = MPC.get_ephemeris('Ceres')

In [30]:
type(a['Uncertainty 3sig', 'Unc. P.A.'])

astropy.table.table.Table

In [ ]:
# If getting errors or things breaking, try clearing cache
MPC.clear_cache()

In [6]:
# Data extraction from JPL is done with the jplhorizons package from astroquery
from astroquery.jplhorizons import Horizons
import astropy.units as u

In [ ]:
# To do anything, it is necessary to instantiate a Horizons object

# You must give an id (int or str, it is the target object), a location (observer, can be geocentric or topocentric or even from another planet) and an epoch. 
obj = Horizons(id='Ceres', location='568', epochs=2458133.33546)

# Here is an example with topocentric observer (it has to be a dictionary with lon, lat and elev, it is vital to use units as well). You can also define a body of reference (the default is earth), but if you need to do it from another planet, there is the option:
statue_of_liberty = {'lon': -74.0466891 * u.deg,
                     'lat': 40.6892534 * u.deg,
                     'elevation': 0.093 * u.km}
obj = Horizons(id='Ceres',
               location=statue_of_liberty,
               epochs=2458133.33546)

# Regarding epochs, it can be a singular value in julian days or a list of values in julian days. You can also ask for a range of dates using the "start" and "stop", but this time using the standard date format “YYYY-MM-DD [HH:MM:SS]” with a "step". By default, epochs is None, which corresponds to the current date and time. 
obj = Horizons(id='Ceres', location='568',
               epochs={'start':'2010-01-01', 'stop':'2010-03-01',
                       'step':'10d'})

# You can also use id_type to control the way JPL searches for the data. None searches everything, prioritizing major bodies, then small bodies. "smallbody" limits the search to comets and asteroids in the solar system. "designation" limits the search to small body designations. "name" limits the search to asteroid or comet names. "asteroid_name" limits the search to asteroid names only. "comet_name" limits the search to comet names only.
# If for any reason you search for a name and get multiple results, just print the object and search again using the specific record / id.
# You can use horizons objects to get ephemerides, orbital elements and state vectors. Here we will focus on ephemerides. 

JPLHorizons instance "Ceres"; location={'lon': <Quantity -74.0466891 deg>, 'lat': <Quantity 40.6892534 deg>, 'elevation': <Quantity 0.093 km>, 'body': 399}, epochs=[2458133.33546], id_type=None


In [ ]:
# To get ephemerides, use the ephemerides method

# Take into account that the ephemerides will be of those of the given epoch from the given location for the target id
obj = Horizons(id='Ceres', location='568',
               epochs={'start':'2010-01-01', 'stop':'2010-03-01',
                       'step':'10d'})
eph = obj.ephemerides()
print(eph)

# There are several optional parameters tho, I will list them for you to consider (from the documentation):
# "(...) airmass_lessthan sets an upper limit to airmass, solar_elongation enables the definition of a solar elongation range, max_hour_angle sets a cutoff of the hour angle, skip_daylight=True rejects epochs during daylight, rate_cutoff rejects targets with sky motion rates higher than provided (in units of arcsec/h), refraction accounts for refraction in the computation of the ephemerides (disabled by default), and refsystem defines the coordinate reference system used (ICRF by default). For comets, the options closest_apparition and no_fragments are available, which selects the closest apparition in time and limits fragment matching (73P-B would only match 73P-B), respectively. Note that these options should only be used for comets and will crash the query for other object types. Extra precision in the queried properties can be requested using the extra_precision option. Furthermore, get_query_payload=True skips the query and only returns the query payload. To pass additional settings to the request use the optional_settings passing a key-value dictionary."

# Most importantly, you can query only the necessary parameters using the quantities property:
eph2 = obj.ephemerides(quantities=1) # Only query RA and DEC
eph3 = obj.ephemerides(quantities='19,20') # Only query heliocentric and geocentric distances
print(eph2)

<TableColumns names=('targetname','datetime_str','datetime_jd','H','G','solar_presence','lunar_presence','r','r_rate','delta','delta_rate')>


In [ ]:
# A good thing to note is that you can get orbital elements using the elements method. However, it is most likely better to just use MPC for this. 

In [1]:
# For physical parameters, it is necessary to use the SBDB API from JPL
from astroquery.jplsbdb import SBDB

In [5]:
# You can search the information using the query method
sbdb = SBDB.query('1927 LA') # Receives number, name or designation
sbdb

OrderedDict([('object',
              OrderedDict([('spkid', '50246923'),
                           ('fullname', '(1927 LA)'),
                           ('orbit_class',
                            OrderedDict([('name', 'Outer Main-belt Asteroid'),
                                         ('code', 'OMB')])),
                           ('des', '1927 LA'),
                           ('prefix', None),
                           ('kind', 'au'),
                           ('neo', False),
                           ('orbit_id', '11'),
                           ('pha', False)])),
             ('orbit',
              OrderedDict([('not_valid_before', None),
                           ('comment', None),
                           ('data_arc', '34'),
                           ('n_dop_obs_used', None),
                           ('moid', <Quantity 1.24 AU>),
                           ('last_obs', '1927-07-05'),
                           ('two_body', None),
                           ('cov_ep